# 🍺 Chopp & Cia · Inteligência de Risco em Comodato
# 🔧 Notebook 03 — Preparação e Congelamento do Split

**Projeto Integrador VI** · 2º Semestre/2026 · FATEC Votorantim

---

## 🎯 Responsabilidade única deste notebook

Transformar o dataset consolidado no **dataset de treino**, definir a variável-alvo e
**congelar a partição treino/teste** numa tabela Delta.

| Entrada | Saída |
| :--- | :--- |
| `dataset_consolidado_v<X>` | `dataset_split_<SPLIT_HASH>` · `dataset_producao_<SPLIT_HASH>` |

## 🧊 Por que o split é congelado numa tabela

Os três notebooks de modelo (04, 05, 06) leem **as mesmas linhas de treino e as mesmas
linhas de teste**. Sem isso, comparar Regressão com Random Forest é comparar dois
modelos avaliados em conjuntos diferentes — e a diferença de métrica pode vir do
sorteio, não do algoritmo.

A alternativa seria cada notebook refazer `train_test_split(random_state=42)`. Parece
equivalente, mas não é: basta uma linha a mais na origem, uma ordenação diferente ou
um filtro alterado para o sorteio mudar em silêncio, e a comparação deixa de ser
pareada sem que nada avise.

Com o split materializado:

- os três modelos são comparáveis por construção;
- cada notebook roda isolado, em qualquer ordem, quantas vezes quiser;
- meses depois, `SPLIT_HASH` recupera exatamente as linhas que produziram as métricas.

## 🔑 As três identidades que amarram um experimento

| Hash | Cobre | Responde |
| :--- | :--- | :--- |
| `DATA_VERSION` | a carga do banco (notebook 01) | *quais dados?* |
| `SPLIT_HASH` | universo + alvo + partição (este notebook) | *quais linhas, com qual rótulo?* |
| `CONFIG_HASH` | hiperparâmetros do modelo (notebooks 04-06) | *qual configuração?* |

Todo run do MLflow carrega os três. Uma diferença de métrica sempre tem endereço.

## 🏗️ Etapas

| # | Etapa | Produz |
| :---: | :--- | :--- |
| 0 | Parametrização | caminhos e catálogo, sem nada fixo no código |
| 1 | Painel de preparação | parâmetros + `SPLIT_HASH` |
| 2 | Carga e contrato | `df_clientes` |
| 3 | Universo de modelagem | `df_ml_completo` (lookup de produção) |
| 4 | Variável-alvo + guarda de vazamento | `df_treino` |
| 5 | Partição estratificada | `X_train`/`X_test`/`y_train`/`y_test` |
| 6 | Materialização em Delta | tabelas de split e produção |
| 7 | Registro no MLflow | run de preparação com o lineage |

## 🎛️ Parâmetros (sem caminhos fixos)

Nenhum caminho de arquivo está escrito no código deste notebook. A origem dos
parâmetros depende de onde ele roda:

| Ambiente | Mecanismo | Onde aparece |
| :--- | :--- | :--- |
| **Databricks** | `dbutils.widgets` | campos no **topo** do notebook |
| **Local** (VS Code / Jupyter) | diálogo do sistema | janela de seleção de arquivo |


### Parâmetros deste notebook

| Parâmetro | O que é | Local | Databricks |
| :--- | :--- | :--- | :--- |
| `catalogo` · `schema` | onde as tabelas vivem | *não usado* | campo de texto |
| `data_version` | qual carga do notebook 01 usar | *não usado* | campo de texto |
| `csv_consolidado` | o CSV do notebook 01 | seletor de arquivo | *não usado* |
| `output_dir` | pasta do split e da produção | seletor de pasta | *não usado* |


### Como funciona localmente

Na primeira execução abre-se o diálogo do Windows. A escolha fica memorizada em
`~/.chopp_risco_params.json`, então **as execuções seguintes não perguntam nada** — o
diálogo só reaparece se o arquivo tiver sido movido, ou se você definir
`FORCAR_SELECAO = True`.

O diálogo é a **caixa nativa do Windows** (`comdlg32`), a mesma do Explorer — não o
`tkinter`. A diferença importa: o tkinter precisa criar uma janela-mãe para ancorar o
diálogo, e dentro do kernel do Jupyter essa janela nasce sem foco e atrás do editor.
A API nativa não cria janela nenhuma.

### 🔧 Se ainda assim o seletor não abrir

Em algumas instalações a janela simplesmente não aparece. Não é preciso lutar com
ela: preencha `CAMINHOS_MANUAIS` no topo da célula e o diálogo deixa de ser usado.

```python
CAMINHOS_MANUAIS = {
    "sql_file": r"C:\dados\DB_POWER_SYS.sql",
}
```

O `r` antes das aspas é necessário para que a barra invertida do Windows não seja
lida como caractere de escape. O que estiver preenchido ali tem **precedência sobre
tudo** — sobre o cache e sobre o diálogo.

### Como funciona no Databricks

Os campos aparecem no topo do notebook assim que a célula roda pela primeira vez.
Preencha e re-execute. Não há seletor de arquivo em notebook do Databricks — o caminho
do Volume é digitado, no formato `/Volumes/<catálogo>/<schema>/<volume>/arquivo`.



> **Sobre segurança:** tirar o caminho do código resolve **portabilidade** (o notebook
> roda na máquina de qualquer pessoa do grupo) e evita expor a estrutura de diretórios
> num repositório público. Não é um controle de acesso: quem executa o notebook lê o
> mesmo arquivo de qualquer forma. O controle de acesso real, no Databricks, vem das
> permissões do Unity Catalog sobre o Volume e as tabelas.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PARAMETRIZAÇÃO SEM CAMINHOS FIXOS
#
#  Nenhum caminho de arquivo é escrito no código. A origem dos parâmetros
#  depende de onde o notebook roda:
#
#    Databricks → dbutils.widgets, os campos que aparecem no topo do notebook.
#                 É o mecanismo nativo da plataforma; não existe file picker
#                 em notebook do Databricks.
#    Local      → caixa de diálogo NATIVA do Windows (a mesma do Explorer),
#                 com a escolha memorizada num JSON. Sem tkinter: ele cria uma
#                 janela-mãe que, dentro do kernel do Jupyter, nasce sem foco e
#                 atrás do VS Code — o diálogo abria, mas ficava invisível.
#
#  O cache local existe para que a segunda execução não reabra o diálogo: ele
#  só volta a aparecer se o arquivo tiver sumido ou se você pedir explicitamente
#  com FORCAR_SELECAO = True.
# ══════════════════════════════════════════════════════════════════════════════
import os
import json
from pathlib import Path

# Colocar em True reabre os diálogos mesmo havendo escolha memorizada.
# No Databricks não tem efeito: lá os widgets já são visíveis e editáveis.
FORCAR_SELECAO = True

# ── Caminhos informados à mão (alternativa ao diálogo) ────────────────────────
#
#  Se por qualquer motivo o seletor gráfico não abrir na sua máquina, preencha
#  aqui e o diálogo deixa de ser necessário. O que estiver preenchido TEM
#  PRECEDÊNCIA sobre tudo: sobre o cache e sobre o diálogo.
#
#  Deixe como está (strings vazias) para usar o seletor gráfico normalmente.
#
#  Use string "crua" (o r antes das aspas) para que a barra invertida do
#  Windows não seja interpretada como escape:
#      CAMINHOS_MANUAIS = {"sql_file": r"C:\dados\DB_POWER.sql"}
CAMINHOS_MANUAIS = {
    # "sql_file":         r"",
    # "output_dir":       r"",
    # "csv_consolidado":  r"",
    # "csv_split":        r"",
}

try:
    dbutils                                     # type: ignore # noqa: F821
    NO_DATABRICKS = True
except NameError:
    NO_DATABRICKS = False

# Onde a escolha local fica memorizada. Fica ao lado do notebook, e não no
# diretório de dados: é preferência de máquina, não dado do projeto.
_CACHE_PARAMS = Path.home() / ".chopp_risco_params.json"


def _cache_ler() -> dict:
    """Lê as escolhas memorizadas. Cache corrompido não pode derrubar o notebook."""
    try:
        if _CACHE_PARAMS.exists():
            return json.loads(_CACHE_PARAMS.read_text(encoding="utf-8"))
    except Exception:
        pass
    return {}


def _cache_gravar(chave: str, valor: str) -> None:
    """Memoriza uma escolha. Falha de escrita é irrelevante — só perde o atalho."""
    try:
        d = _cache_ler()
        d[chave] = str(valor)
        _CACHE_PARAMS.write_text(
            json.dumps(d, indent=2, ensure_ascii=False), encoding="utf-8"
        )
    except Exception:
        pass


# ══════════════════════════════════════════════════════════════════════════════
#  DIÁLOGO NATIVO DO WINDOWS (ctypes → comdlg32)
#
#  Por que não tkinter: o tkinter cria uma janela-mãe (Tk) para ancorar o
#  diálogo, e dentro do kernel do Jupyter essa janela nasce sem foco e atrás do
#  VS Code. O diálogo abre — mas fica invisível, e a célula parece travada.
#
#  A API abaixo é a mesma que o Explorer e qualquer programa Windows usam.
#  Não cria janela nenhuma: não há o que ficar atrás de nada.
# ══════════════════════════════════════════════════════════════════════════════
def _dialogo_windows_arquivo(titulo: str, filtros, inicial=None):
    """
    Caixa de seleção de arquivo nativa (GetOpenFileNameW). Caminho ou None.

    filtros: lista [(rótulo, padrão), ...] — ex.: [("Dump SQL", "*.sql")]
    """
    import ctypes
    from ctypes import wintypes

    class OPENFILENAMEW(ctypes.Structure):
        _fields_ = [
            ("lStructSize", wintypes.DWORD), ("hwndOwner", wintypes.HWND),
            ("hInstance", wintypes.HINSTANCE), ("lpstrFilter", wintypes.LPCWSTR),
            ("lpstrCustomFilter", wintypes.LPWSTR), ("nMaxCustFilter", wintypes.DWORD),
            ("nFilterIndex", wintypes.DWORD), ("lpstrFile", wintypes.LPWSTR),
            ("nMaxFile", wintypes.DWORD), ("lpstrFileTitle", wintypes.LPWSTR),
            ("nMaxFileTitle", wintypes.DWORD), ("lpstrInitialDir", wintypes.LPCWSTR),
            ("lpstrTitle", wintypes.LPCWSTR), ("Flags", wintypes.DWORD),
            ("nFileOffset", wintypes.WORD), ("nFileExtension", wintypes.WORD),
            ("lpstrDefExt", wintypes.LPCWSTR), ("lCustData", wintypes.LPARAM),
            ("lpfnHook", wintypes.LPVOID), ("lpTemplateName", wintypes.LPCWSTR),
            ("pvReserved", wintypes.LPVOID), ("dwReserved", wintypes.DWORD),
            ("FlagsEx", wintypes.DWORD),
        ]

    buf = ctypes.create_unicode_buffer(4096)

    # A API espera pares "rótulo\0padrão\0", terminados por um \0 extra.
    partes = []
    for rotulo, padrao in filtros:
        partes += [rotulo, padrao]
    filtro_api = "\0".join(partes) + "\0\0"

    ofn = OPENFILENAMEW()
    ofn.lStructSize = ctypes.sizeof(OPENFILENAMEW)
    ofn.hwndOwner = ctypes.windll.user32.GetForegroundWindow()
    ofn.lpstrFilter = filtro_api
    ofn.lpstrFile = ctypes.cast(buf, wintypes.LPWSTR)
    ofn.nMaxFile = 4096
    ofn.lpstrTitle = titulo
    ofn.lpstrInitialDir = inicial
    # NOCHANGEDIR: sem isto o diálogo muda o diretório de trabalho do kernel,
    # e caminhos relativos usados depois passam a apontar para outro lugar.
    ofn.Flags = 0x00001000 | 0x00000800 | 0x00000008 | 0x00080000

    if ctypes.windll.comdlg32.GetOpenFileNameW(ctypes.byref(ofn)):
        return buf.value or None
    return None


def _dialogo_windows_pasta(titulo: str):
    """Caixa de seleção de pasta nativa (SHBrowseForFolderW). Caminho ou None."""
    import ctypes
    from ctypes import wintypes

    class BROWSEINFOW(ctypes.Structure):
        _fields_ = [
            ("hwndOwner", wintypes.HWND), ("pidlRoot", ctypes.c_void_p),
            ("pszDisplayName", wintypes.LPWSTR), ("lpszTitle", wintypes.LPCWSTR),
            ("ulFlags", wintypes.UINT), ("lpfn", wintypes.LPVOID),
            ("lParam", wintypes.LPARAM), ("iImage", ctypes.c_int),
        ]

    shell32 = ctypes.windll.shell32
    nome = ctypes.create_unicode_buffer(4096)

    bi = BROWSEINFOW()
    bi.hwndOwner = ctypes.windll.user32.GetForegroundWindow()
    bi.pszDisplayName = ctypes.cast(nome, wintypes.LPWSTR)
    bi.lpszTitle = titulo
    bi.ulFlags = 0x00000001 | 0x00000040     # só diretórios + diálogo moderno

    shell32.SHBrowseForFolderW.restype = ctypes.c_void_p
    pidl = shell32.SHBrowseForFolderW(ctypes.byref(bi))
    if not pidl:
        return None
    try:
        caminho = ctypes.create_unicode_buffer(4096)
        shell32.SHGetPathFromIDListW.argtypes = [ctypes.c_void_p, wintypes.LPWSTR]
        ok = shell32.SHGetPathFromIDListW(pidl, caminho)
        return caminho.value if ok else None
    finally:
        # A lista de IDs é alocada pelo shell; liberá-la é responsabilidade nossa.
        ctypes.windll.ole32.CoTaskMemFree(ctypes.c_void_p(pidl))


def _dialogo_tkinter_arquivo(titulo: str, tipos, inicial=None):
    """Alternativa por tkinter, para quando a API do Windows não estiver disponível."""
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.update_idletasks()
    root.attributes("-topmost", True)
    root.lift()
    try:
        root.focus_force()
    except Exception:
        pass
    root.update()
    try:
        return filedialog.askopenfilename(
            parent=root, title=titulo, filetypes=tipos,
            initialdir=inicial or str(Path.home()),
        ) or None
    finally:
        try:
            root.destroy()
        except Exception:
            pass


def _dialogo_tkinter_pasta(titulo: str, inicial=None):
    """Alternativa por tkinter para seleção de pasta."""
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.update_idletasks()
    root.attributes("-topmost", True)
    root.lift()
    try:
        root.focus_force()
    except Exception:
        pass
    root.update()
    try:
        return filedialog.askdirectory(
            parent=root, title=titulo, initialdir=inicial or str(Path.home())
        ) or None
    finally:
        try:
            root.destroy()
        except Exception:
            pass


def _selecionar_arquivo(titulo: str, tipos, inicial=None):
    """
    Abre a caixa de seleção de arquivo. Retorna o caminho ou None.

    Tenta primeiro a API nativa do Windows (sem janela intermediária) e recorre
    ao tkinter fora do Windows ou se a API falhar.
    """
    if os.name == "nt":
        try:
            return _dialogo_windows_arquivo(titulo, tipos, inicial)
        except Exception as e:
            print(f"   ⚠️  Diálogo nativo falhou ({type(e).__name__}: {e});"
                  f" tentando tkinter...")

    try:
        return _dialogo_tkinter_arquivo(titulo, tipos, inicial)
    except ImportError:
        print("   ⚠️  Nenhum seletor disponível (sem tkinter).")
        return None
    except Exception as e:
        print(f"   ⚠️  Seletor indisponível ({type(e).__name__}: {e}).")
        return None


def _selecionar_pasta(titulo: str, inicial=None):
    """Abre a caixa de seleção de pasta. Retorna o caminho ou None."""
    if os.name == "nt":
        try:
            return _dialogo_windows_pasta(titulo)
        except Exception as e:
            print(f"   ⚠️  Diálogo nativo falhou ({type(e).__name__}: {e});"
                  f" tentando tkinter...")

    try:
        return _dialogo_tkinter_pasta(titulo, inicial)
    except ImportError:
        print("   ⚠️  Nenhum seletor disponível (sem tkinter).")
        return None
    except Exception as e:
        print(f"   ⚠️  Seletor indisponível ({type(e).__name__}: {e}).")
        return None


def widget(nome: str, padrao: str = "", rotulo: str = None,
           opcoes: list = None) -> str:
    """
    Declara um parâmetro de texto (ou dropdown) e devolve seu valor.

    No Databricks vira um campo no topo do notebook. Localmente devolve o valor
    memorizado, ou o padrão.

    A recriação do widget a cada execução é intencional: mudar o padrão no
    código passa a valer sem precisar remover o widget à mão.
    """
    rotulo = rotulo or nome
    if NO_DATABRICKS:
        try:
            if opcoes:
                dbutils.widgets.dropdown(nome, padrao or opcoes[0],
                                         opcoes, rotulo)          # noqa: F821
            else:
                dbutils.widgets.text(nome, padrao, rotulo)        # noqa: F821
        except Exception:
            pass   # widget já existe com outro tipo — o valor abaixo ainda serve
        try:
            return dbutils.widgets.get(nome)                      # noqa: F821
        except Exception:
            return padrao
    return _cache_ler().get(nome, padrao)


def parametro_arquivo(nome: str, titulo: str, tipos, padrao_databricks: str = "",
                      rotulo: str = None) -> str:
    """
    Resolve o caminho de um ARQUIVO de entrada.

    Databricks → widget de texto (caminho do Volume; não há file picker lá).
    Local      → escolha memorizada, ou diálogo do sistema.

    Falha com mensagem clara se nada for escolhido: seguir com caminho vazio
    produziria um FileNotFoundError muito adiante, sem contexto.
    """
    if NO_DATABRICKS:
        valor = widget(nome, padrao_databricks, rotulo or titulo)
        if not valor:
            raise ValueError(
                f"\n\n  ❌ Parâmetro '{nome}' vazio.\n\n"
                f"     Preencha o campo '{rotulo or titulo}' no topo do notebook\n"
                f"     com o caminho do arquivo no Volume, por exemplo:\n"
                f"       /Volumes/<catalogo>/<schema>/<volume>/arquivo.sql\n"
            )
        return valor

    # 1. Caminho informado à mão vence tudo — é a saída para quando o diálogo
    #    gráfico não abre.
    manual = (CAMINHOS_MANUAIS.get(nome) or "").strip()
    if manual:
        if not os.path.exists(manual):
            raise FileNotFoundError(
                f"\n\n  ❌ CAMINHOS_MANUAIS['{nome}'] aponta para um arquivo que\n"
                f"     não existe:\n\n       {manual}\n\n"
                f"     Corrija o caminho no topo desta célula.\n"
            )
        print(f"   ✍️  {nome}: caminho informado em CAMINHOS_MANUAIS")
        print(f"      {manual}")
        return manual

    # 2. Escolha memorizada de uma execução anterior
    memorizado = _cache_ler().get(nome)
    if memorizado and os.path.exists(memorizado) and not FORCAR_SELECAO:
        print(f"   📎 {nome}: usando a escolha memorizada")
        print(f"      {memorizado}")
        print(f"      (para escolher outro, defina FORCAR_SELECAO = True acima)")
        return memorizado

    if memorizado and not os.path.exists(memorizado):
        print(f"   ⚠️  O arquivo memorizado não existe mais:")
        print(f"      {memorizado}")

    # 3. Diálogo gráfico
    print(f"   📂 Abrindo o seletor de arquivo...")
    print(f"      ⚠️  A janela pode abrir ATRÁS do editor — procure na barra de tarefas.")
    escolhido = _selecionar_arquivo(
        titulo, tipos,
        inicial=os.path.dirname(memorizado) if memorizado else None,
    )
    if not escolhido:
        raise ValueError(
            f"\n\n  ❌ Nenhum arquivo selecionado para '{nome}'.\n\n"
            f"     Isso acontece se você cancelou o diálogo — ou se ele não chegou\n"
            f"     a aparecer (em algumas instalações do VS Code a janela do\n"
            f"     tkinter nasce sem foco).\n\n"
            f"     DUAS SAÍDAS:\n\n"
            f"     (a) informe o caminho à mão — preencha no topo desta célula:\n"
            f"           CAMINHOS_MANUAIS = {{\n"
            f"               \"{nome}\": r\"C:\\caminho\\para\\o\\arquivo\",\n"
            f"           }}\n"
            f"         e re-execute. É a opção que não depende de janela nenhuma.\n\n"
            f"     (b) re-execute a célula e procure a janela na barra de tarefas\n"
            f"         (ícone do Python) antes de clicar em qualquer outro lugar.\n"
        )
    _cache_gravar(nome, escolhido)
    print(f"   ✅ Selecionado e memorizado para as próximas execuções.")
    return escolhido


def parametro_pasta(nome: str, titulo: str, padrao_databricks: str = "",
                    rotulo: str = None) -> str:
    """
    Resolve o caminho de um DIRETÓRIO de saída.

    Diferença em relação a parametro_arquivo(): um diretório inexistente é
    criado em vez de recusado — é saída, não entrada.
    """
    if NO_DATABRICKS:
        valor = widget(nome, padrao_databricks, rotulo or titulo)
        if not valor:
            raise ValueError(
                f"\n\n  ❌ Parâmetro '{nome}' vazio.\n"
                f"     Preencha o campo '{rotulo or titulo}' no topo do notebook.\n"
            )
        return valor

    # 1. Caminho informado à mão vence tudo
    manual = (CAMINHOS_MANUAIS.get(nome) or "").strip()
    if manual:
        os.makedirs(manual, exist_ok=True)
        print(f"   ✍️  {nome}: caminho informado em CAMINHOS_MANUAIS")
        print(f"      {manual}")
        return manual

    # 2. Escolha memorizada
    memorizado = _cache_ler().get(nome)
    if memorizado and not FORCAR_SELECAO:
        os.makedirs(memorizado, exist_ok=True)
        print(f"   📎 {nome}: usando a escolha memorizada")
        print(f"      {memorizado}")
        return memorizado

    # 3. Diálogo gráfico
    print(f"   📂 Abrindo o seletor de pasta...")
    print(f"      ⚠️  A janela pode abrir ATRÁS do editor — procure na barra de tarefas.")
    escolhido = _selecionar_pasta(titulo, inicial=memorizado)
    if not escolhido:
        raise ValueError(
            f"\n\n  ❌ Nenhuma pasta selecionada para '{nome}'.\n\n"
            f"     Isso acontece se você cancelou o diálogo — ou se ele não chegou\n"
            f"     a aparecer (em algumas instalações do VS Code a janela do\n"
            f"     tkinter nasce sem foco).\n\n"
            f"     DUAS SAÍDAS:\n\n"
            f"     (a) informe o caminho à mão — preencha no topo desta célula:\n"
            f"           CAMINHOS_MANUAIS = {{\n"
            f"               \"{nome}\": r\"C:\\caminho\\para\\a\\pasta\",\n"
            f"           }}\n"
            f"         e re-execute. É a opção que não depende de janela nenhuma.\n\n"
            f"     (b) re-execute a célula e procure a janela na barra de tarefas.\n"
        )
    os.makedirs(escolhido, exist_ok=True)
    _cache_gravar(nome, escolhido)
    print(f"   ✅ Selecionada e memorizada para as próximas execuções.")
    return escolhido


print("✅ Parametrização carregada")
print(f"   ├─ Ambiente : {'Databricks (widgets)' if NO_DATABRICKS else 'Local (seletor de arquivo)'}")
if not NO_DATABRICKS:
    print(f"   └─ Cache    : {_CACHE_PARAMS}")
else:
    print(f"   └─ Os parâmetros aparecem como campos no TOPO do notebook.")

# A seleção local acontece nesta própria célula.
CSV_ENTRADA = None if NO_DATABRICKS else parametro_arquivo(
    nome="csv_consolidado",
    titulo="Selecione o dataset consolidado (CSV do notebook 01)",
    tipos=[("CSV", "*.csv"), ("Todos os arquivos", "*.*")],
)

## 🎛️ 1. Painel de Preparação

Este painel governa **o que o modelo vê** — não como ele aprende. A separação é
deliberada:

| Aqui (notebook 03) | Lá (notebooks 04-06) |
| :--- | :--- |
| quem é elegível (`min_compras`) | qual `C`, qual `max_depth` |
| o que é "alto risco" (`alvo`) | quantos folds de CV |
| quais colunas são feature | qual scaler, qual encoder |
| como a partição é sorteada | qual métrica elege o campeão |

Mudar algo aqui **muda o problema** e produz um `SPLIT_HASH` novo — todos os modelos
precisam ser re-treinados para continuarem comparáveis. Mudar algo lá muda apenas
aquele modelo.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PAINEL DE PREPARAÇÃO
#  Governa O QUE o modelo vê. Como ele aprende é assunto dos notebooks 04-06.
# ══════════════════════════════════════════════════════════════════════════════
import os
import json
import hashlib
import warnings
from datetime import datetime
from typing import Dict, Any, List

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import display

import mlflow

# NO_DATABRICKS, widget(), parametro_arquivo() e parametro_pasta() vêm da
# célula de parametrização acima. Ela precisa ter sido executada.
if "parametro_arquivo" not in dir():
    raise NameError(
        "Execute a célula de PARAMETRIZAÇÃO (logo acima) antes desta. "
        "É ela que resolve os caminhos sem deixá-los fixos no código."
    )


# ══════════════════════════════════════════════════════════════════════════════
#  ⬇️  PARÂMETROS QUE VOCÊ AJUSTA  ⬇️
# ══════════════════════════════════════════════════════════════════════════════

# ── Origem e destino, sem caminho fixo ───────────────────────────────────────
# Databricks → campos no topo do notebook.
# Local      → diálogo de seleção na primeira execução, memorizado depois.
_CATALOGO = widget("catalogo", "projetointegrador", "Catálogo (Unity Catalog)")
_SCHEMA = widget("schema", "projetointegrador", "Schema")
_DATA_VERSION_ALVO = widget("data_version", "1.0", "Versão dos dados de entrada")

# Tabela publicada pelo notebook 01. O sufixo é a DATA_VERSION.
TABELA_ENTRADA = (
    f"{_CATALOGO}.{_SCHEMA}.dataset_consolidado_v{_DATA_VERSION_ALVO.replace('.', '_')}"
)

# Destino das tabelas produzidas aqui — mesmo catálogo e schema da entrada.
DESTINO = {"catalogo": _CATALOGO, "schema": _SCHEMA}

# Fora do Databricks: CSV de entrada e pasta de saída, via diálogo.
if NO_DATABRICKS:
    DIR_SAIDA_LOCAL = "."
else:
    DIR_SAIDA_LOCAL = parametro_pasta(
        nome="output_dir",
        titulo="Selecione a pasta de saída (split e produção)",
    )

LEITURA_CSV = {"sep": ";", "encoding": "utf-8-sig", "low_memory": False}

# Semente única, propagada ao split. Não é hiperparâmetro: é dispositivo de
# REPRODUTIBILIDADE. Variá-la para "melhorar" a métrica é escolher o sorteio
# favorável, não um modelo melhor.
RANDOM_STATE = 42


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 1 · UNIVERSO DE MODELAGEM  —  quais LINHAS entram
# ══════════════════════════════════════════════════════════════════════════════
UNIVERSO: Dict[str, Any] = {

    # Recorte de negócio: o projeto é sobre chopp e chopeira, não sobre a
    # operação inteira. A flag vem pronta do notebook 01, marcada no grão
    # transacional (depende de ID_PRODUTO, que não sobrevive à agregação).
    "filtro_core_business": True,

    # Elegibilidade (cold-start). A taxa de atraso de um cliente com 1 parcela só
    # pode valer 0% ou 100% — entra no dataset com a mesma cara de uma taxa
    # calculada sobre 200 parcelas.
    #
    # ⚠️ EXCLUSIVO: FREQUENCIA_COMPRAS > min_compras.
    #    min_compras=2 mantém quem tem 3 ou mais compras.
    #
    # 📊 A seção 3.7 do notebook 02 mede o custo de cada valor; o notebook 04
    #    varre esta grade como experimento (ESTUDO='elegibilidade').
    "min_compras": 2,

    # Recorte temporal opcional: mantém só clientes cuja última compra é
    # posterior a esta data. None = sem recorte.
    # Serve ao estudo de janela temporal — carteira recente generaliza melhor?
    "data_corte_ultima_compra": None,   # ex.: "2024-01-01"
}


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 2 · VARIÁVEL-ALVO  —  o que o modelo aprende a prever
#
#  Não é parâmetro estatístico: é REGRA DE NEGÓCIO. Mudar limite_atraso ou
#  combinador redefine o problema, e métricas de rodadas com definições
#  diferentes NÃO são comparáveis — são problemas distintos.
# ══════════════════════════════════════════════════════════════════════════════
ALVO: Dict[str, Any] = {
    "coluna": "ALTO_RISCO",
    "limite_atraso": 0.20,      # > 20% de atrasos ⇒ ALTO_RISCO
    "combinador": "OU",         # OU | E  — atraso financeiro __ de comodato
}


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 3 · FEATURES  —  quais COLUNAS o modelo vê
# ══════════════════════════════════════════════════════════════════════════════
FEATURES: Dict[str, Any] = {

    "numericas": [
        "FREQUENCIA_COMPRAS",
        "TICKET_MEDIO",
        "TOTAL_GASTO",
        "DIAS_DESDE_PRIMEIRA_COMPRA",
        "DIAS_DESDE_ULTIMA_COMPRA",
        "TOTAL_PARCELAS",
        "TOTAL_COMODATOS",
        # ⚠️ As duas abaixo compartilham origem aritmética com o alvo:
        #    MEDIA_DIAS_ATRASO_* e TAXA_ATRASO_* medem o mesmo fenômeno.
        #    Mantê-las produz AUC alto que mede capacidade de REPRODUZIR a regra,
        #    não de prever. Removê-las é o experimento de ablação — disponível
        #    como ESTUDO='ablacao' nos notebooks 04-06.
        "MEDIA_DIAS_ATRASO_PAG",
        "MEDIA_DIAS_ATRASO_COM",
    ],

    "categoricas": [
        "PERFIL",
        "CIDADE",
        "PAGAMENTO",
        # SEGMENTO fica fora: a EDA 3.1 mostrou que o cadastro é ruidoso
        # (comércios classificados como "Consumidor Final"). Treinar sobre um
        # rótulo sabidamente errado ensina o erro ao modelo.
    ],

    # Colunas que existem no dataset e NUNCA podem entrar no modelo: ou são
    # identidade, ou são o próprio alvo disfarçado. A guarda da célula 4 falha
    # se alguma delas aparecer na lista de features.
    #
    # NOME_CLIENTE e DS_FANTASIA não constam mais aqui porque deixaram de
    # existir: saíram do contrato de colunas do notebook 01 no contrato de colunas. A guarda
    # abaixo cobre o caso de alguém reintroduzi-las numa carga futura.
    "proibidas": [
        "ID_PESSOA",
        "NOME_CLIENTE", "DS_FANTASIA", "NM_PESSOA",        # identificação nominal
        "TAXA_ATRASO_PAGAMENTO", "TAXA_ATRASO_COMODATO",   # constroem o alvo
        "PARCELAS_ATRASADAS", "COMODATOS_ATRASADOS",       # numeradores do alvo
        "RISCO_FINANCEIRO", "RISCO_COMODATO", "PERFIL_RISCO",  # o alvo em outra escala
        "MAX_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_COM",      # base do aging
        "AGING_PAGAMENTO", "AGING_COMODATO",
    ],

    # Viaja junto do dataset para rastreabilidade, mas nunca entra no fit.
    # Só a chave: nesta arquitetura não há identificação nominal no pipeline.
    "identidade": ["ID_PESSOA"],
}


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 4 · PARTIÇÃO TREINO/TESTE
# ══════════════════════════════════════════════════════════════════════════════
SPLIT: Dict[str, Any] = {
    "test_size": 0.30,
    "stratify": True,     # obrigatório: a classe positiva é fortemente majoritária
    "shuffle": True,
}


# ── Derivados ─────────────────────────────────────────────────────────────────
FEATURES_NUM: List[str] = FEATURES["numericas"]
FEATURES_CAT: List[str] = FEATURES["categoricas"]
ALL_FEATURES: List[str] = FEATURES_NUM + FEATURES_CAT
TARGET: str = ALVO["coluna"]


def calcular_hash(*blocos, tamanho: int = 12) -> str:
    """SHA-256 truncado de um JSON canônico dos blocos (chaves ordenadas)."""
    canonico = json.dumps(blocos, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(canonico.encode("utf-8")).hexdigest()[:tamanho]


# ══════════════════════════════════════════════════════════════════════════════
#  SPLIT_HASH — a identidade do dataset de modelagem
#
#  Cobre tudo que define QUAIS LINHAS, COM QUAL RÓTULO e EM QUAL PARTIÇÃO.
#  Inclui a tabela de entrada: o mesmo filtro sobre outra carga é outro dataset.
#  Dois experimentos com o mesmo SPLIT_HASH são comparáveis entre si; com hashes
#  diferentes, não são — e o painel do MLflow deixa isso explícito.
# ══════════════════════════════════════════════════════════════════════════════
SPLIT_HASH = calcular_hash(TABELA_ENTRADA, UNIVERSO, ALVO, FEATURES, SPLIT, RANDOM_STATE)

TABELA_SPLIT = f"{DESTINO['catalogo']}.{DESTINO['schema']}.dataset_split_{SPLIT_HASH}"
TABELA_PRODUCAO = f"{DESTINO['catalogo']}.{DESTINO['schema']}.dataset_producao_{SPLIT_HASH}"
TABELA_CATALOGO_SPLITS = f"{DESTINO['catalogo']}.{DESTINO['schema']}.catalogo_splits"

# Experimento do MLflow — o mesmo dos notebooks 04-06, para que tudo apareça
# num painel só.
EXPERIMENT_NAME = "/Users/brunofsaraujo@hotmail.com/Chopp_Cia_Experimentos"

print("=" * 78)
print(f"{'PAINEL DE PREPARAÇÃO':^78}")
print("=" * 78)
print(f"  Entrada        : {TABELA_ENTRADA if NO_DATABRICKS else os.path.basename(CSV_ENTRADA)}")
print(f"  SPLIT_HASH     : {SPLIT_HASH}")
print("-" * 78)
print(f"  UNIVERSO")
print(f"    ├─ core business  : {UNIVERSO['filtro_core_business']}")
print(f"    ├─ min_compras    : > {UNIVERSO['min_compras']} (exclusivo)")
print(f"    └─ corte temporal : {UNIVERSO['data_corte_ultima_compra'] or 'sem recorte'}")
print(f"  ALVO")
print(f"    └─ {TARGET} = atraso > {ALVO['limite_atraso']:.0%} ({ALVO['combinador']})")
print(f"  FEATURES")
print(f"    └─ {len(ALL_FEATURES)} ({len(FEATURES_NUM)} num + {len(FEATURES_CAT)} cat)")
print(f"  PARTIÇÃO")
print(f"    └─ {int((1-SPLIT['test_size'])*100)}/{int(SPLIT['test_size']*100)} "
      f"estratificado, semente {RANDOM_STATE}")
print("-" * 78)
print(f"  Tabelas a publicar:")
print(f"    ├─ {TABELA_SPLIT}")
print(f"    └─ {TABELA_PRODUCAO}")
print("=" * 78)

## 📂 2. Carga e Verificação do Contrato

A validação aqui é mais rígida do que na EDA: além das colunas existirem, as features
declaradas no painel precisam estar presentes **e sem NaN**. Um `NaN` numérico só
falha lá na frente, dentro do `StandardScaler`, com uma mensagem que não aponta para
a causa.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CARGA DO DATASET CONSOLIDADO
# ══════════════════════════════════════════════════════════════════════════════
print(f"📂 Carregando {TABELA_ENTRADA if NO_DATABRICKS else os.path.basename(CSV_ENTRADA)}...\n")

if NO_DATABRICKS:
    df_clientes = spark.read.table(TABELA_ENTRADA).toPandas()        # noqa: F821
else:
    df_clientes = pd.read_csv(CSV_ENTRADA, **LEITURA_CSV)

# ── Normalização de tipos ─────────────────────────────────────────────────────
# Mantida mesmo lendo de Delta (que já traz schema): esta célula precisa produzir
# o MESMO df vindo da tabela ou do CSV, senão as duas origens divergem em
# silêncio e o resultado depende de onde o notebook rodou.
for _col in ["PRIMEIRA_COMPRA", "ULTIMA_COMPRA"]:
    if _col in df_clientes.columns:
        df_clientes[_col] = pd.to_datetime(df_clientes[_col], errors="coerce")

_COLS_NUM = [
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA", "FREQUENCIA_COMPRAS",
    "TOTAL_ITENS", "QTD_TOTAL_VENDIDA", "TOTAL_GASTO", "TICKET_MEDIO",
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "TAXA_ATRASO_PAGAMENTO",
    "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG", "VALOR_TOTAL_PARCELAS",
    "TOTAL_COMODATOS", "COMODATOS_ATRASADOS", "TAXA_ATRASO_COMODATO",
    "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM", "QTD_EQUIPAMENTOS",
    "RISCO_FINANCEIRO", "RISCO_COMODATO",
    "CORE_BUSINESS", "TEM_VENDAS", "TEM_FINANCEIRO", "TEM_COMODATO",
]
for _col in _COLS_NUM:
    if _col in df_clientes.columns:
        df_clientes[_col] = pd.to_numeric(df_clientes[_col], errors="coerce").fillna(0)

# ── Contrato: as colunas de que ESTE notebook depende ────────────────────────
_necessarias = set(ALL_FEATURES) | {
    "ID_PESSOA", "CORE_BUSINESS", "FREQUENCIA_COMPRAS",
    "TAXA_ATRASO_PAGAMENTO", "TAXA_ATRASO_COMODATO",
}
_ausentes = sorted(_necessarias - set(df_clientes.columns))
if _ausentes:
    raise KeyError(
        f"Colunas ausentes em {TABELA_ENTRADA}: {_ausentes}\n"
        f"   Se você adicionou uma feature ao painel, acrescente a coluna de\n"
        f"   origem em COLUNAS_NECESSARIAS (notebook 01), incremente\n"
        f"   DATA_VERSION e reexecute a ingestão."
    )

if df_clientes["ID_PESSOA"].duplicated().any():
    raise ValueError("ID_PESSOA duplicado — a tabela deveria ter 1 linha por cliente.")

DATA_VERSION_LIDA = (
    str(df_clientes["_DATA_VERSION"].iloc[0])
    if "_DATA_VERSION" in df_clientes.columns else "desconhecida"
)

print(f"   ✅ {df_clientes.shape[0]:,} clientes × {df_clientes.shape[1]} colunas")
print(f"   └─ Versão dos dados: {DATA_VERSION_LIDA}")

## 🎯 3. Universo de Modelagem

Dois dataframes com propósitos distintos, e a distinção importa:

| | `df_ml_completo` | `df_treino` |
| :--- | :--- | :--- |
| **Quem** | todos os clientes do recorte de negócio | só os elegíveis |
| **Identidade** | tem `ID_PESSOA` | a chave viaja mas nunca entra no fit |
| **Alvo** | não tem | tem `ALTO_RISCO` |
| **Para quê** | *lookup* de produção — "qual o risco do cliente 1042?" | treinar e avaliar |

Um cliente com uma única compra não deve treinar o modelo — a taxa dele é ruído — mas
**deve poder ser escorado** em produção. Por isso `df_ml_completo` é mais amplo que
`df_treino`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  UNIVERSO DE MODELAGEM
#  Cada filtro é reportado com o que custou: um filtro que remove 80% da base é
#  uma decisão de projeto, não um detalhe de implementação.
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 78)
print(f"{'🎯 CONSTRUÇÃO DO UNIVERSO DE MODELAGEM':^78}")
print("=" * 78)

_n0 = len(df_clientes)
df_ml_completo = df_clientes.copy()
print(f"\n  Base consolidada                     : {_n0:>6,} clientes")

# ── Filtro 1: core business ───────────────────────────────────────────────────
if UNIVERSO["filtro_core_business"]:
    if "CORE_BUSINESS" not in df_ml_completo.columns:
        raise KeyError(
            "Coluna CORE_BUSINESS ausente — reexecute o notebook 01. Sem ela não "
            "há como isolar os clientes de chopp/chopeira."
        )
    _n = len(df_ml_completo)
    df_ml_completo = df_ml_completo[df_ml_completo["CORE_BUSINESS"] == 1].copy()
    print(f"  ├─ core business (chopp/chopeira)    : {len(df_ml_completo):>6,} "
          f"({-(_n - len(df_ml_completo)):+,})")

# ── Filtro 2: recorte temporal (opcional) ─────────────────────────────────────
# Serve ao estudo de janela: uma carteira recente generaliza melhor que a base
# histórica inteira? O notebook 04 varre este eixo como experimento.
if UNIVERSO["data_corte_ultima_compra"]:
    _corte = pd.to_datetime(UNIVERSO["data_corte_ultima_compra"])
    _n = len(df_ml_completo)
    df_ml_completo = df_ml_completo[
        pd.to_datetime(df_ml_completo["ULTIMA_COMPRA"], errors="coerce") >= _corte
    ].copy()
    print(f"  ├─ última compra ≥ {_corte:%d/%m/%Y}       : {len(df_ml_completo):>6,} "
          f"({-(_n - len(df_ml_completo)):+,})")

# ── Tratamento de ausentes ────────────────────────────────────────────────────
# Zero é a leitura correta para contagens e taxas: "nenhuma parcela" são 0
# parcelas, não um valor desconhecido.
_COLS_ZERO = [c for c in [
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG",
    "TAXA_ATRASO_PAGAMENTO", "TOTAL_COMODATOS", "COMODATOS_ATRASADOS",
    "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM", "TAXA_ATRASO_COMODATO",
    "QTD_EQUIPAMENTOS",
] if c in df_ml_completo.columns]
df_ml_completo[_COLS_ZERO] = df_ml_completo[_COLS_ZERO].fillna(0)

for _col in FEATURES_CAT:
    df_ml_completo[_col] = df_ml_completo[_col].fillna("NÃO INFORMADO")
for _col in FEATURES["identidade"]:
    if _col in df_ml_completo.columns and df_ml_completo[_col].dtype == object:
        df_ml_completo[_col] = df_ml_completo[_col].fillna("N/D")

# ── NaN residual em feature numérica ──────────────────────────────────────────
# Falha aqui com mensagem clara; sem isso, o erro aparece dentro do
# StandardScaler, dezenas de células adiante, apontando para outro lugar.
_com_nan = [c for c in FEATURES_NUM if df_ml_completo[c].isna().any()]
if _com_nan:
    raise ValueError(
        f"Features numéricas com NaN após o tratamento: {_com_nan}.\n"
        f"   Verifique a agregação no notebook 01 — uma coluna que deveria ser 0\n"
        f"   está chegando como ausente."
    )

print(f"\n  ✅ df_ml_completo (lookup de produção): {len(df_ml_completo):,} clientes")
print(f"     └─ {len(ALL_FEATURES)} features + {len(FEATURES['identidade'])} colunas de identidade")

print(f"\n  Distribuição das categóricas:")
for _cat in FEATURES_CAT:
    _vc = df_ml_completo[_cat].value_counts()
    _top = ", ".join(f"{k}={v}" for k, v in _vc.head(3).items())
    print(f"    ├─ {_cat:<12} ({_vc.size} valores): {_top}")

## 🎲 4. Variável-Alvo e Guarda de Vazamento

O alvo é construído a partir de `TAXA_ATRASO_PAGAMENTO` e `TAXA_ATRASO_COMODATO`.
Justamente por isso, essas duas colunas — e os numeradores que as compõem — **não
podem entrar como feature**. A checagem é barata e evita o erro mais caro possível:
um AUC alto que não significa nada.

> ⚠️ **Vazamento parcial reconhecido.** `MEDIA_DIAS_ATRASO_PAG` e
> `MEDIA_DIAS_ATRASO_COM` continuam entre as features e compartilham origem
> aritmética com o alvo. A escolha é consciente e está documentada em
> `LIMITACOES_CONHECIDAS`: as métricas medem a capacidade de **reproduzir a regra de
> negócio**, não de prever o futuro. O experimento de ablação (`ESTUDO='ablacao'`,
> notebooks 04-06) quantifica exatamente quanto do desempenho vem daí.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  VARIÁVEL-ALVO
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 78)
print(f"{'🎲 VARIÁVEL-ALVO E ELEGIBILIDADE':^78}")
print("=" * 78)

# ── Filtro de elegibilidade ───────────────────────────────────────────────────
# EXCLUSIVO: min_compras=2 mantém quem tem 3 ou mais compras.
_min = UNIVERSO["min_compras"]
df_treino = df_ml_completo[df_ml_completo["FREQUENCIA_COMPRAS"] > _min].copy()

print(f"\n  df_ml_completo                : {len(df_ml_completo):>6,} clientes")
print(f"  df_treino (compras > {_min})       : {len(df_treino):>6,} clientes")
print(f"  Descartados por histórico     : {len(df_ml_completo)-len(df_treino):>6,} "
      f"({(len(df_ml_completo)-len(df_treino))/max(len(df_ml_completo),1)*100:.1f}%)")

if len(df_treino) < 50:
    raise ValueError(
        f"Apenas {len(df_treino)} clientes elegíveis — amostra pequena demais para "
        f"treinar. Reduza UNIVERSO['min_compras'] ou revise o recorte temporal."
    )

# ── Construção do alvo ────────────────────────────────────────────────────────
_lim = ALVO["limite_atraso"]
_cond_pag = df_treino["TAXA_ATRASO_PAGAMENTO"] > _lim
_cond_com = df_treino["TAXA_ATRASO_COMODATO"] > _lim

if ALVO["combinador"] == "OU":
    _condicao = _cond_pag | _cond_com
elif ALVO["combinador"] == "E":
    _condicao = _cond_pag & _cond_com
else:
    raise ValueError(f"ALVO['combinador']='{ALVO['combinador']}' inválido — use 'OU' ou 'E'.")

df_treino[TARGET] = _condicao.astype(int)

# ══════════════════════════════════════════════════════════════════════════════
#  GUARDA 1 · IDENTIFICAÇÃO NOMINAL
#  Verificada ANTES do vazamento porque é um problema DIFERENTE, com uma causa
#  diferente — e a mensagem precisa dizer qual dos dois é.
#
#  Nome de cliente não prevê inadimplência. Um modelo com acesso a ele aprende
#  CLIENTES em vez de COMPORTAMENTO, e passa a discriminar por identidade onde
#  deveria discriminar por conduta. Num modelo de crédito, é o viés que não se
#  pode cometer.
#
#  Por decisão de projeto essas colunas não são sequer extraídas do ERP (notebook 01).
#  A guarda existe para o caso de uma carga futura reintroduzi-las: sem ela, a
#  coluna voltaria em silêncio e nada avisaria.
# ══════════════════════════════════════════════════════════════════════════════
_PADROES_NOMINAIS = ("NOME", "NM_", "FANTASIA", "RAZAO", "RAZÃO",
                     "CPF", "CNPJ", "EMAIL", "TELEFONE", "ENDERECO", "ENDEREÇO")
_nominais_em_features = sorted(
    c for c in ALL_FEATURES
    if any(p in c.upper() for p in _PADROES_NOMINAIS)
)
if _nominais_em_features:
    raise ValueError(
        f"❌ IDENTIFICAÇÃO NOMINAL COMO FEATURE: {_nominais_em_features}\n\n"
        f"   Colunas que identificam a PESSOA não podem treinar o modelo: ele\n"
        f"   aprenderia clientes específicos em vez de comportamento de risco,\n"
        f"   discriminando por identidade em vez de por conduta.\n\n"
        f"   Isto NÃO é vazamento do alvo — é viés de identificação. A coluna\n"
        f"   não constrói ALTO_RISCO; ela simplesmente não deveria existir aqui.\n\n"
        f"   Remova-as de FEATURES['numericas'/'categoricas'] no painel."
    )

# ══════════════════════════════════════════════════════════════════════════════
#  GUARDA 2 · VAZAMENTO DO ALVO
#  As colunas que CONSTROEM o alvo não podem entrar como feature. Sem esta
#  checagem o notebook roda até o fim, entrega AUC alto e não significa nada.
# ══════════════════════════════════════════════════════════════════════════════
_vazamento = sorted(set(FEATURES["proibidas"]) & set(ALL_FEATURES))
if _vazamento:
    raise ValueError(
        f"❌ VAZAMENTO DO ALVO: as features {_vazamento} estão na lista de\n"
        f"   proibidas — elas constroem ALTO_RISCO. Um modelo treinado com elas\n"
        f"   apenas recalcula a própria regra e reporta AUC ~1,0."
    )

# As mesmas colunas também não devem viajar no dataset materializado. Avisamos
# em vez de falhar: a presença no dataset de origem não contamina o modelo, mas
# indica que o contrato do notebook 01 mudou sem que este notebook soubesse.
_nominais_no_dataset = sorted(
    c for c in df_treino.columns
    if any(p in c.upper() for p in _PADROES_NOMINAIS)
)
if _nominais_no_dataset:
    print(f"\n  ⚠️  O dataset de origem traz colunas de identificação nominal:")
    print(f"      {_nominais_no_dataset}")
    print(f"      Elas NÃO entram no modelo (a guarda acima garante), mas desde a")
    print(f"      não deveriam existir. Verifique COLUNAS_NECESSARIAS no")
    print(f"      notebook 01 — elas serão excluídas das tabelas publicadas aqui.")

if len(set(ALL_FEATURES)) != len(ALL_FEATURES):
    _dups = [c for c in set(ALL_FEATURES) if ALL_FEATURES.count(c) > 1]
    raise ValueError(f"Features duplicadas entre numéricas e categóricas: {_dups}")

_faltando = [c for c in ALL_FEATURES if c not in df_treino.columns]
if _faltando:
    raise KeyError(f"Features declaradas no painel e ausentes em df_treino: {_faltando}")

# ── Distribuição do alvo ──────────────────────────────────────────────────────
n_total = len(df_treino)
n_risco = int(df_treino[TARGET].sum())
n_bom = n_total - n_risco
pct_risco = n_risco / n_total * 100

print(f"\n{'─'*78}")
print(f"  Regra: TAXA_ATRASO_PAG > {_lim:.0%}  {ALVO['combinador']}  TAXA_ATRASO_COM > {_lim:.0%}")
print(f"{'─'*78}")
print(f"  Alto risco  (1) : {n_risco:>6,}  ({pct_risco:>5.1f}%)")
print(f"  Bom pagador (0) : {n_bom:>6,}  ({100-pct_risco:>5.1f}%)")

# A classe minoritária é que determina o que se pode medir. Com poucos casos, a
# métrica de holdout tem intervalo de confiança largo demais para sustentar
# comparação entre modelos.
_minoria = min(n_risco, n_bom)
_classe_min = "bom pagador (0)" if n_bom < n_risco else "alto risco (1)"

# ── Classe degenerada: não há problema de classificação a resolver ───────────
# Um alvo constante não é "muito desbalanceado": não há o que aprender. O
# StratifiedKFold falharia adiante com mensagem sobre número de membros por
# classe, que não aponta para a causa — que está na REGRA DO ALVO, não na CV.
if _minoria == 0:
    raise ValueError(
        f"\n\n  ❌ ALVO DEGENERADO: todos os {n_total} clientes elegíveis caíram na\n"
        f"     mesma classe ({'alto risco' if n_bom == 0 else 'bom pagador'}).\n\n"
        f"     Não há problema de classificação a resolver, e os notebooks 04-06\n"
        f"     falhariam na validação cruzada com uma mensagem que não aponta\n"
        f"     para a causa real — que está aqui, na definição do alvo.\n\n"
        f"     Regra atual: TAXA_ATRASO_PAG > {_lim:.0%} "
        f"{ALVO['combinador']} TAXA_ATRASO_COM > {_lim:.0%}\n\n"
        f"     Caminhos possíveis:\n"
        f"       (a) ajuste ALVO['limite_atraso'] — o limite atual não separa esta\n"
        f"           carteira (veja a distribuição na seção 3.4 do notebook 02)\n"
        f"       (b) troque ALVO['combinador'] de 'OU' para 'E' — 'OU' é bem mais\n"
        f"           permissivo e satura quando as duas taxas são altas\n"
        f"       (c) reveja o filtro de elegibilidade: um corte agressivo pode ter\n"
        f"           deixado só a cauda de inadimplentes\n"
    )

print(f"\n  ⚠️  Classe minoritária: {_classe_min} com {_minoria} casos")
_n_min_prev = int(_minoria * SPLIT["test_size"])
print(f"      → cerca de {_n_min_prev} cairão no conjunto de teste.")
if _n_min_prev < 20:
    print(f"      → abaixo de 20: os intervalos de confiança serão largos e a")
    print(f"        seleção de campeão deve usar 'cv', não 'teste'.")

# A CV estratificada exige ao menos n_splits membros da classe rara. Avisar
# aqui é muito mais útil que o ValueError que o sklearn lança lá adiante.
if _minoria < 5:
    print(f"      → ⚠️  com {_minoria} casos, uma CV de 5 folds falhará nos")
    print(f"           notebooks 04-06 (precisa de ao menos 1 por fold).")

if pct_risco > 70 or pct_risco < 30:
    print(f"\n  ⚠️  Forte desbalanceamento ({pct_risco:.0f}% positivos):")
    print(f"      · use class_weight='balanced' nos notebooks 04-06")
    print(f"      · accuracy e AUC enganam — leia MCC e balanced accuracy")
    print(f"      · compare SEMPRE contra o baseline de classe majoritária")

## ✂️ 5. Partição Treino/Teste

Estratificada e com semente fixa. O `n` da classe minoritária no teste é destacado
porque **é ele que governa a largura de todos os intervalos de confiança** dos
notebooks seguintes — com poucos casos, um único cliente reclassificado move a
especificidade vários pontos percentuais.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PARTIÇÃO TREINO / TESTE
# ══════════════════════════════════════════════════════════════════════════════
X = df_treino[ALL_FEATURES].copy()
y = df_treino[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=SPLIT["test_size"],
    random_state=RANDOM_STATE,
    shuffle=SPLIT["shuffle"],
    stratify=y if SPLIT["stratify"] else None,
)

# Índices preservados para reconstruir a identidade das linhas na materialização:
# é o que permite responder "quem foi classificado como o quê" na auditoria.
IDX_TRAIN, IDX_TEST = X_train.index, X_test.index

n_neg_teste = int((y_test == 0).sum())
n_pos_teste = int((y_test == 1).sum())

print("=" * 78)
print(f"{'✂️  PARTIÇÃO TREINO / TESTE':^78}")
print("=" * 78)
print(f"\n  X_train : {X_train.shape[0]:>5,} clientes  ({int((1-SPLIT['test_size'])*100)}%)")
print(f"  X_test  : {X_test.shape[0]:>5,} clientes  ({int(SPLIT['test_size']*100)}%)")
print(f"  Estratificado: {SPLIT['stratify']}   |   semente: {RANDOM_STATE}")
print(f"\n  Prevalência no treino : {y_train.mean():.1%}")
print(f"  Prevalência no teste  : {y_test.mean():.1%}")
print(f"    └─ diferença: {abs(y_train.mean()-y_test.mean())*100:.2f}p "
      f"(estratificação {'funcionou' if abs(y_train.mean()-y_test.mean()) < 0.02 else 'com desvio'})")

print(f"\n{'─'*78}")
print(f"  ⚠️  COMPOSIÇÃO DO TESTE — governa TODOS os intervalos de confiança")
print(f"{'─'*78}")
print(f"     positivos (alto risco) : {n_pos_teste:>4}")
print(f"     negativos (bom pagador): {n_neg_teste:>4}   ← o que limita a specificity")
_n_min_teste = min(n_pos_teste, n_neg_teste)
if 0 < _n_min_teste < 20:
    print(f"\n     Com {_n_min_teste} casos na classe rara, um cliente reclassificado move")
    print(f"     a métrica ~{100/_n_min_teste:.1f} pontos percentuais. Leia sempre o IC junto")
    print(f"     do valor pontual, e prefira selecao.origem='cv' nos notebooks 04-06.")
elif _n_min_teste == 0:
    # Só alcançável se a estratificação estiver desligada — com ela, um alvo
    # degenerado já teria sido barrado na célula anterior.
    raise ValueError(
        f"O conjunto de teste ficou com 0 casos de uma das classes. "
        f"Nenhuma métrica de holdout é calculável assim. "
        f"Verifique SPLIT['stratify'] (atual: {SPLIT['stratify']}) e o tamanho "
        f"da classe minoritária ({_minoria} no dataset elegível)."
    )

## 💾 6. Materialização em Delta

Duas tabelas gerenciadas no Unity Catalog, ambas com `SPLIT_HASH` no nome:

| Tabela | Conteúdo | Consumida por |
| :--- | :--- | :--- |
| `dataset_split_<hash>` | features + alvo + coluna `_SPLIT` (`train`/`test`) | notebooks 04, 05, 06 |
| `dataset_producao_<hash>` | todos os clientes + identidade, sem alvo | escoragem (`predict_by_id`) |

O hash no nome não é decoração: ele torna impossível um notebook de modelo ler
acidentalmente o split de outra configuração. Se você mudar `min_compras`, o nome da
tabela muda, e o notebook 04 apontando para o nome antigo continua lendo o dataset
antigo — de forma explícita, não silenciosa.

> Uso de `saveAsTable()`, não `save(caminho)`: cria **tabela no catálogo**, consultável
> por SQL e com governança, em vez de arquivos Delta soltos num Volume — que é o que
> fazia aparecerem os `.parquet` na versão anterior.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MATERIALIZAÇÃO EM DELTA
#
#  saveAsTable(), não save(): cria TABELA no catálogo — consultável por SQL,
#  governada por GRANT, com comentários e lineage. `save()` num Volume gravaria
#  arquivos Delta soltos, e o que se vê ao navegar são os part-*.snappy.parquet
#  de dentro do Delta.
# ══════════════════════════════════════════════════════════════════════════════

def _preparar_para_spark(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza um DataFrame pandas para escrita via Spark.

    Spark não aceita nome de coluna com espaço ou pontuação, e coluna 'object'
    só com NaN vira NullType e quebra a escrita.
    """
    out = df.copy()
    out.columns = [
        str(c).strip().replace(" ", "_").replace("-", "_").replace(".", "_")
        for c in out.columns
    ]
    for col in out.columns:
        if out[col].dtype == "object":
            out[col] = out[col].fillna("").astype(str)
    return out


def publicar_tabela(df: pd.DataFrame, nome: str, comentario: str,
                    propriedades: dict) -> str:
    """Publica um DataFrame pandas como tabela gerenciada Delta."""
    if not NO_DATABRICKS:
        caminho = os.path.join(DIR_SAIDA_LOCAL, f"{nome.split('.')[-1]}.csv")
        df.to_csv(caminho, sep=";", encoding="utf-8-sig", index=False)
        print(f"   ✅ {os.path.basename(caminho):<44} {df.shape[0]:>6,} × {df.shape[1]:>3}")
        return caminho

    sdf = spark.createDataFrame(_preparar_para_spark(df))             # noqa: F821
    (
        sdf.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome)
    )

    _props = ", ".join(f"'{k}' = '{str(v).replace(chr(39), chr(39)*2)}'"
                       for k, v in propriedades.items())
    spark.sql(f"ALTER TABLE {nome} SET TBLPROPERTIES ({_props})")      # noqa: F821
    spark.sql(                                                        # noqa: F821
        f"COMMENT ON TABLE {nome} IS '{comentario.replace(chr(39), chr(39)*2)}'"
    )
    print(f"   ✅ {nome.split('.')[-1]:<44} {df.shape[0]:>6,} × {df.shape[1]:>3}")
    return nome


print("=" * 78)
print(f"{'💾 MATERIALIZAÇÃO EM DELTA':^78}")
print("=" * 78)
print(f"  SPLIT_HASH: {SPLIT_HASH}\n")

_props_comuns = {
    "split_hash": SPLIT_HASH,
    "data_version": DATA_VERSION_LIDA,
    "tabela_origem": TABELA_ENTRADA,
    "min_compras": UNIVERSO["min_compras"],
    "limite_atraso": ALVO["limite_atraso"],
    "combinador_alvo": ALVO["combinador"],
    "random_state": RANDOM_STATE,
    "gerado_em": datetime.now().isoformat(),
}

# ── 1. Split congelado: o que os notebooks 04-06 leem ────────────────────────
# A chave viaja junto para auditoria ("qual cliente foi classificado como o quê"),
# mas os notebooks de modelo selecionam só ALL_FEATURES para o fit.
_cols_id = [c for c in FEATURES["identidade"] if c in df_treino.columns]

_tr = df_treino.loc[IDX_TRAIN, _cols_id + ALL_FEATURES + [TARGET]].copy()
_tr["_SPLIT"] = "train"
_te = df_treino.loc[IDX_TEST, _cols_id + ALL_FEATURES + [TARGET]].copy()
_te["_SPLIT"] = "test"

df_split = pd.concat([_tr, _te], ignore_index=True)
df_split["_SPLIT_HASH"] = SPLIT_HASH
df_split["_DATA_VERSION"] = DATA_VERSION_LIDA

PATH_SPLIT = publicar_tabela(
    df_split, TABELA_SPLIT,
    comentario=(
        f"Partição treino/teste congelada · split_hash={SPLIT_HASH} · "
        f"origem={TABELA_ENTRADA} · min_compras>{UNIVERSO['min_compras']} · "
        f"alvo={TARGET} (atraso>{ALVO['limite_atraso']:.0%} {ALVO['combinador']}). "
        f"Lida pelos notebooks 04, 05 e 06 — as MESMAS linhas nos três, para que "
        f"a comparação entre algoritmos seja pareada."
    ),
    propriedades={**_props_comuns,
                  "n_train": len(_tr), "n_test": len(_te),
                  "n_test_negativos": n_neg_teste,
                  "prevalencia": round(float(y.mean()), 4)},
)

# ── 2. Base de produção: lookup por ID ───────────────────────────────────────
# Mais ampla que o treino de propósito: um cliente com uma só compra não deve
# treinar o modelo, mas deve poder ser escorado.
_cols_prod = _cols_id + ALL_FEATURES + [
    c for c in ["FREQUENCIA_COMPRAS", "TOTAL_GASTO", "ULTIMA_COMPRA",
                "TAXA_ATRASO_PAGAMENTO", "TAXA_ATRASO_COMODATO", "PERFIL_RISCO"]
    if c in df_ml_completo.columns and c not in ALL_FEATURES
]
df_producao = df_ml_completo[list(dict.fromkeys(_cols_prod))].copy()
df_producao["_SPLIT_HASH"] = SPLIT_HASH
df_producao["_DATA_VERSION"] = DATA_VERSION_LIDA

PATH_PRODUCAO = publicar_tabela(
    df_producao, TABELA_PRODUCAO,
    comentario=(
        f"Base de lookup para escoragem · split_hash={SPLIT_HASH}. TODOS os "
        f"clientes do universo de negócio, identificados por ID_PESSOA e sem alvo. Mais ampla "
        f"que o dataset de treino: cliente com histórico curto não treina o "
        f"modelo, mas pode ser escorado por ele."
    ),
    propriedades={**_props_comuns, "n_clientes": len(df_producao)},
)

print(f"\n  Leitura nos notebooks 04-06:")
print(f'     TABELA_SPLIT = "{TABELA_SPLIT}"')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CATÁLOGO DE SPLITS — o índice dos datasets de modelagem
#  Responde "quais splits existem e em que diferem" com um SELECT.
# ══════════════════════════════════════════════════════════════════════════════
registro_split = {
    "split_hash": SPLIT_HASH,
    "tabela_split": TABELA_SPLIT,
    "tabela_producao": TABELA_PRODUCAO,
    "tabela_origem": TABELA_ENTRADA,
    "data_version": DATA_VERSION_LIDA,
    "min_compras": UNIVERSO["min_compras"],
    "filtro_core_business": str(UNIVERSO["filtro_core_business"]),
    "data_corte_ultima_compra": str(UNIVERSO["data_corte_ultima_compra"]),
    "limite_atraso": ALVO["limite_atraso"],
    "combinador": ALVO["combinador"],
    "n_features": len(ALL_FEATURES),
    "features": ", ".join(ALL_FEATURES),
    "n_train": len(_tr),
    "n_test": len(_te),
    "n_test_negativos": n_neg_teste,
    "prevalencia": round(float(y.mean()), 4),
    "random_state": RANDOM_STATE,
    "gerado_em": datetime.now().isoformat(),
}

if NO_DATABRICKS:
    _sdf = spark.createDataFrame(pd.DataFrame([registro_split]))       # noqa: F821
    (_sdf.write.format("delta").mode("append").option("mergeSchema", "true")
     .saveAsTable(TABELA_CATALOGO_SPLITS))
    print(f"📚 Split registrado em {TABELA_CATALOGO_SPLITS}\n")

    _hist = spark.sql(f"""
        SELECT split_hash, data_version, min_compras, combinador,
               n_train, n_test, n_test_negativos, prevalencia
        FROM {TABELA_CATALOGO_SPLITS}
        ORDER BY gerado_em DESC LIMIT 12
    """).toPandas()                                                   # noqa: F821
    display(_hist)
else:
    _p = os.path.join(DIR_SAIDA_LOCAL, "catalogo_splits.csv")
    _df = pd.DataFrame([registro_split])
    if os.path.exists(_p):
        _df = pd.concat([pd.read_csv(_p, sep=";"), _df], ignore_index=True)
    _df.to_csv(_p, sep=";", encoding="utf-8-sig", index=False)
    print(f"📚 Split registrado em catalogo_splits.csv")

## 📝 7. Registro no MLflow

Um run de preparação, marcado como `etapa='preparacao'`, que guarda a configuração
completa e o lineage. É o que permite responder, ao olhar um modelo campeão meses
depois: *de onde vieram estes dados, com que filtro e com que rótulo?*

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  REGISTRO DA PREPARAÇÃO NO MLFLOW
#  Não treina modelo nenhum: registra a PROVENIÊNCIA do dataset, para que os
#  runs dos notebooks 04-06 possam se referir a ela pelo SPLIT_HASH.
# ══════════════════════════════════════════════════════════════════════════════

# Run órfão da sessão anterior capturaria tudo que viesse depois: mlflow.log_*
# fora de um run não falha — abre um run implícito que nunca é encerrado.
if mlflow.active_run() is not None:
    mlflow.end_run()

try:
    mlflow.set_experiment(EXPERIMENT_NAME)
    _mlflow_ok = True
except Exception as e:
    print(f"⚠️  MLflow indisponível ({type(e).__name__}): {e}")
    print("   As tabelas foram publicadas; só o registro do run foi pulado.")
    _mlflow_ok = False

if _mlflow_ok:
    with mlflow.start_run(run_name=f"PREPARACAO__{SPLIT_HASH}") as run:
        RUN_ID_PREPARACAO = run.info.run_id

        mlflow.set_tags({
            "projeto": "Chopp & Cia",
            "instituicao": "FATEC Votorantim",
            "disciplina": "Projeto Integrador VI",
            "etapa": "preparacao",
            "split_hash": SPLIT_HASH,
            "data_version": DATA_VERSION_LIDA,
            "unidade_analise": "ID_PESSOA",
        })

        # Prefixos deliberados: no painel você filtra por 'univ.' para isolar
        # mudanças de universo e por 'alvo.' para mudanças de definição do alvo.
        mlflow.log_params({
            **{f"univ.{k}": v for k, v in UNIVERSO.items()},
            **{f"alvo.{k}": v for k, v in ALVO.items()},
            **{f"split.{k}": v for k, v in SPLIT.items()},
            "split_hash": SPLIT_HASH,
            "data_version": DATA_VERSION_LIDA,
            "tabela_origem": TABELA_ENTRADA,
            "tabela_split": TABELA_SPLIT,
            "tabela_producao": TABELA_PRODUCAO,
            "random_state": RANDOM_STATE,
            "n_features_total": len(ALL_FEATURES),
            "n_features_num": len(FEATURES_NUM),
            "n_features_cat": len(FEATURES_CAT),
            "features": ", ".join(ALL_FEATURES),
        })

        mlflow.log_metrics({
            "n_base_consolidada": float(_n0),
            "n_universo": float(len(df_ml_completo)),
            "n_elegiveis": float(len(df_treino)),
            "n_train": float(len(X_train)),
            "n_test": float(len(X_test)),
            "n_test_negativos": float(n_neg_teste),
            "n_test_positivos": float(n_pos_teste),
            "prevalencia": float(y.mean()),
            "prevalencia_train": float(y_train.mean()),
            "prevalencia_test": float(y_test.mean()),
        })

        mlflow.log_dict({
            "split_hash": SPLIT_HASH,
            "data_version": DATA_VERSION_LIDA,
            "tabela_origem": TABELA_ENTRADA,
            "universo": UNIVERSO,
            "alvo": ALVO,
            "features": FEATURES,
            "split": SPLIT,
            "random_state": RANDOM_STATE,
            "tabelas_publicadas": {
                "split": TABELA_SPLIT,
                "producao": TABELA_PRODUCAO,
            },
            "gerado_em": datetime.now().isoformat(),
        }, "config_preparacao.json")

        # Limitações que acompanham este dataset e devem ser lidas junto de
        # qualquer métrica produzida a partir dele.
        mlflow.log_dict({
            "sem_corte_temporal": (
                "Features e alvo sao calculados sobre a MESMA janela temporal. O "
                "desenho correto exigiria features ate uma data de corte e alvo "
                "observado depois dela. Nao aplicado por tamanho de amostra. "
                "Consequencia: as metricas medem capacidade de REPRODUZIR a regra "
                "de negocio, nao de prever o futuro."
            ),
            "features_derivadas_do_alvo": (
                "MEDIA_DIAS_ATRASO_PAG/COM compartilham origem aritmetica com "
                "TAXA_ATRASO_*, que constroi ALTO_RISCO. Quantificado pelo "
                "experimento ESTUDO='ablacao' nos notebooks 04-06."
            ),
            "classe_positiva_majoritaria": (
                f"{y.mean():.1%} dos clientes elegiveis sao ALTO_RISCO. Acuracia e "
                f"AUC isoladas enganam; compare sempre contra o baseline."
            ),
            "n_pequeno_no_teste": (
                f"Apenas {min(n_pos_teste, n_neg_teste)} casos da classe minoritaria "
                f"no holdout. Intervalos de confianca largos; prefira selecao por CV."
            ),
        }, "limitacoes_metodologicas.json")

    print(f"✅ Preparação registrada no MLflow")
    print(f"   ├─ Experimento : {EXPERIMENT_NAME}")
    print(f"   └─ Run ID      : {RUN_ID_PREPARACAO}")

In [ ]:
# ─── Resumo final ─────────────────────────────────────────────────────────────
print("\n" + "=" * 78)
print(f"{'🎉 PREPARAÇÃO CONCLUÍDA':^78}")
print("=" * 78)
print(f"  SPLIT_HASH   : {SPLIT_HASH}")
print(f"  DATA_VERSION : {DATA_VERSION_LIDA}")
print("-" * 78)
print(f"  Base consolidada     : {_n0:>6,} clientes")
print(f"  Universo de negócio  : {len(df_ml_completo):>6,} clientes  (core business)")
print(f"  Elegíveis ao treino  : {len(df_treino):>6,} clientes  (compras > {UNIVERSO['min_compras']})")
print(f"    ├─ treino          : {len(X_train):>6,}")
print(f"    └─ teste           : {len(X_test):>6,}  ({n_neg_teste} negativos)")
print(f"  Prevalência do alvo  : {y.mean():>6.1%}")
print("-" * 78)
print("  TABELAS PUBLICADAS")
print(f"    ├─ {TABELA_SPLIT}")
print(f"    └─ {TABELA_PRODUCAO}")
print("=" * 78)
print()
print("  ➜ PRÓXIMO PASSO — nos notebooks 04, 05 e 06, configure:")
print()
print(f'       TABELA_SPLIT = "{TABELA_SPLIT}"')
print()
print("     Os três leem as MESMAS linhas de treino e teste. É isso que torna a")
print("     comparação entre Regressão, Árvore e Random Forest pareada: qualquer")
print("     diferença de métrica vem do algoritmo, não do sorteio da partição.")
print()
print("  ⚠️  Mudou algo neste painel? O SPLIT_HASH muda, novas tabelas são criadas,")
print("      e os três notebooks precisam apontar para o novo nome — senão")
print("      continuam medindo o dataset antigo.")
print("=" * 78)